# DPO: hands-on (optional)
**LLMs, Animated · Lesson M3.4**

You'll write Direct Preference Optimization from scratch in PyTorch (CPU is plenty): the loss, its gradient, the sequence log-probabilities it needs, and finally a tiny policy trained with DPO. You'll check that it lands exactly on the policy RLHF would have found, without ever training a reward model.

Each exercise has a `# TODO` cell and a **check** cell with `assert`s. Solutions are at the bottom; try first.

Needs `torch` and `matplotlib` (both pre-installed on Colab). Runs in well under a minute.

In [ ]:
import math, itertools
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
torch.set_num_threads(1)   # tiny tensors: one thread is fastest
print("torch", torch.__version__)

## 1 · The DPO loss
For a prompt $x$, chosen response $y_w$ and rejected response $y_l$:

$$\mathcal{L}_\text{DPO} = -\log \sigma\Big(\underbrace{\beta\big[\log\pi_\theta(y_w) - \log\pi_\text{ref}(y_w)\big]}_{\text{implicit reward of } y_w} - \underbrace{\beta\big[\log\pi_\theta(y_l) - \log\pi_\text{ref}(y_l)\big]}_{\text{implicit reward of } y_l}\Big)$$

In real training you get the four log-probs from two forward passes (policy and frozen reference) over each response. Here we pass them in directly.

In [ ]:
def dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1):
    """Each input: a tensor of shape [B] with log π(y | x) of a WHOLE response.
    pi_*: policy log-probs (trainable), ref_*: reference log-probs (frozen).
    Returns (mean loss, implicit reward of the chosen responses [B], implicit reward of the rejected [B])."""
    # TODO: implicit rewards: beta * (policy log-prob − reference log-prob), for chosen and for rejected
    # TODO: loss = −log σ(chosen reward − rejected reward), averaged over the batch. Use F.logsigmoid (stable).
    raise NotImplementedError("Exercise 1")

**Check 1.** The lesson's worked example, its faded example, and the very first step of training.

In [ ]:
t = lambda *v: torch.tensor(v, dtype=torch.float64)
loss, rw, rl = dpo_loss(t(-12.0), t(-15.0), t(-14.0), t(-13.0), beta=0.1)
assert torch.allclose(rw, t(0.2)) and torch.allclose(rl, t(-0.2))
assert abs(loss.item() - 0.5130) < 1e-4
loss2, _, _ = dpo_loss(t(-20.0), t(-18.0), t(-21.0), t(-17.5), beta=0.5)
assert abs(loss2.item() - 0.3869) < 1e-4
loss0, _, _ = dpo_loss(t(-3.0, -7.0), t(-4.0, -1.0), t(-3.0, -7.0), t(-4.0, -1.0), beta=0.3)
assert abs(loss0.item() - math.log(2)) < 1e-9, "policy == reference gives log 2, whatever β is"
print("✓ exercise 1")

## 2 · Which way does the gradient push?
Let autograd compute the gradient of the worked example with respect to the policy's two log-probs. Then predict it from the lesson's formula.

In [ ]:
pi_w = torch.tensor([-12.0], requires_grad=True)
pi_l = torch.tensor([-15.0], requires_grad=True)
loss, _, _ = dpo_loss(pi_w, pi_l, torch.tensor([-14.0]), torch.tensor([-13.0]), beta=0.1)
loss.backward()
print(f"dL/dlog π(y_w) = {pi_w.grad.item():+.4f}    dL/dlog π(y_l) = {pi_l.grad.item():+.4f}")

In [ ]:
# TODO: fill these in from the lesson
grad_w_formula = None          # ∂L/∂log π(y_w) = −β · σ(−β · margin); here β = 0.1 and the log-ratio margin is 4.0
descent_raises_chosen = None   # True or False: does a gradient-DESCENT step increase log π(y_w)?

**Check 2**

In [ ]:
assert abs(pi_w.grad.item() - grad_w_formula) < 1e-6
assert abs(pi_l.grad.item() + pi_w.grad.item()) < 1e-9, "the push on y_l is equal and opposite"
assert descent_raises_chosen is True
print("✓ exercise 2: descent raises log π(y_w) and lowers log π(y_l)")

## 3 · Log-probability of a whole response
$\log\pi(y \mid x)$ is the **sum** of the response tokens' log-probs, with the prompt masked, exactly like the SFT loss mask (M3.1). Implement it for a batch.

In [ ]:
def sequence_logprob(logits, targets, mask):
    """logits:  [B, T, V], where logits[:, t] is the model's prediction for targets[:, t]
    targets: [B, T] token ids
    mask:    [B, T], 1 for response tokens and 0 for prompt / padding
    Returns  [B]: the sum of the log-probs of the response tokens."""
    # TODO: log_softmax over the vocabulary, pick each target's log-prob with torch.gather,
    #       zero out the masked positions, and sum over T
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
B, T, V = 2, 5, 7
targets = torch.randint(0, V, (B, T))
mask = torch.tensor([[0, 0, 1, 1, 1], [0, 1, 1, 0, 0]], dtype=torch.float32)   # prompt / response / padding
uniform = sequence_logprob(torch.zeros(B, T, V), targets, mask)
assert torch.allclose(uniform, mask.sum(-1) * math.log(1 / V)), "uniform logits: each response token costs log(1/V)"
logits = torch.randn(B, T, V)
manual = torch.stack([sum(F.log_softmax(logits[b, t], -1)[targets[b, t]] for t in range(T) if mask[b, t]) for b in range(B)])
assert torch.allclose(sequence_logprob(logits, targets, mask), manual, atol=1e-5)
changed = logits.clone(); changed[:, 0] += 10.0          # mess with a prompt position
assert torch.allclose(sequence_logprob(changed, targets, mask), sequence_logprob(logits, targets, mask)), "masked positions must not count"
print("✓ exercise 3")

## 4 · Train a tiny policy with DPO
One prompt, four candidate answers. The "policy" is a softmax over 4 learnable logits, starting **as** the reference. Hidden true rewards decide how often people prefer one answer over another (Bradley–Terry). We use the exact preference probabilities $p_{ij}$ as **soft labels**, as if we had asked infinitely many people.

Theory from the lesson: the best KL-regularized RLHF policy is $\pi^*(y) \propto \pi_\text{ref}(y)\, e^{r(y)/\beta}$. If DPO really is RLHF in disguise, training on preferences alone should land on $\pi^*$, with no reward model anywhere.

In [ ]:
ref_logits = torch.tensor([0.0, 0.8, 0.3, -0.2])     # the "SFT model" over the 4 answers
r_true = torch.tensor([1.0, -0.5, 0.5, -1.0])        # hidden: answer 0 is best, 3 worst
pairs = torch.tensor(list(itertools.combinations(range(4), 2)))
i, j = pairs[:, 0], pairs[:, 1]                        # all 6 pairs (i, j)
p_ij = torch.sigmoid(r_true[i] - r_true[j])            # P(answer i beats answer j)
ref_lp = F.log_softmax(ref_logits, -1)
print("reference π_ref:", F.softmax(ref_logits, -1).numpy().round(3))

In [ ]:
def train_toy_dpo(beta, steps=400, lr=0.05, labels=None):
    """labels: P(answer i beats answer j) for every pair (defaults to the soft labels p_ij)."""
    labels = p_ij if labels is None else labels
    theta = torch.nn.Parameter(ref_logits.clone())     # the policy starts AS the reference
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(steps):
        lp = F.log_softmax(theta, -1)                  # log π(answer) for all 4 answers
        # TODO 1: log-ratio margin for every pair: m = (lp[i] - ref_lp[i]) - (lp[j] - ref_lp[j])
        # TODO 2: soft-label DPO loss: -(labels * F.logsigmoid(beta * m) + (1 - labels) * F.logsigmoid(-beta * m)).mean()
        # TODO 3: opt.zero_grad(); loss.backward(); opt.step()
        raise NotImplementedError("Exercise 4")
    return F.softmax(theta.detach(), -1)

**Check 4**

In [ ]:
beta = 0.5
pi = train_toy_dpo(beta)
pi_star = F.softmax(ref_lp + r_true / beta, -1)        # RLHF's closed-form optimum
print("DPO policy:", pi.numpy().round(3))
print("π* (RLHF) :", pi_star.numpy().round(3))
assert (pi - pi_star).abs().max() < 1e-3, "DPO should recover the RLHF-optimal policy"
assert pi.argmax().item() == 0
print("✓ exercise 4: no reward model was trained, yet DPO found RLHF's optimum")

## Experiment
1. **β is the leash.** The cell below trains with several β and plots $\mathrm{KL}(\pi \,\|\, \pi_\text{ref})$. Which way does it go, and why does that match the role of β in the RLHF objective?
2. **Hard labels.** Now pretend every person always picks the truly better answer ($p_{ij} \in \{0, 1\}$). Train longer and longer. Does the policy settle anywhere? This is the over-fitting problem IPO (Azar et al., 2023) was designed to fix.

In [ ]:
def kl(p, q_log):
    return (p * (p.log() - q_log)).sum().item()

betas = [0.25, 0.5, 1.0, 2.0]
kls = [kl(train_toy_dpo(b), ref_lp) for b in betas]
hard = (r_true[i] > r_true[j]).float()
steps_list = [50, 200, 800]
best = [train_toy_dpo(0.5, steps=s, labels=hard)[0].item() for s in steps_list]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(betas, kls, "o-"); ax[0].set_xscale("log"); ax[0].set_xlabel("β"); ax[0].set_ylabel("KL(π ‖ π_ref)")
ax[0].set_title("Soft labels: larger β stays closer")
ax[1].plot(steps_list, best, "o-"); ax[1].set_xscale("log"); ax[1].set_xlabel("training steps"); ax[1].set_ylabel("π(best answer)")
ax[1].set_title("Hard labels: keeps sharpening")
plt.tight_layout(); plt.show()
print("KL by β:", dict(zip(betas, [round(k, 3) for k in kls])))
print("π(best) with hard labels:", dict(zip(steps_list, [round(b, 3) for b in best])))

## Recap
- DPO loss $= -\log\sigma(\beta[\text{log-ratio}(y_w) - \text{log-ratio}(y_l)])$: four log-probs, one sigmoid.
- Its gradient raises $\log\pi(y_w)$, lowers $\log\pi(y_l)$, weighted by $\sigma(-\text{margin})$.
- $\log\pi(y\mid x)$ is the masked sum of response-token log-probs.
- With enough (soft) preference data, DPO recovers the KL-regularized RLHF optimum $\pi_\text{ref}\,e^{r/\beta}$; larger β stays closer to the reference.
- With hard, deterministic labels it keeps sharpening; brief training (or IPO-style losses) is the practical leash.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1):
    r_w = beta * (pi_w - ref_w)
    r_l = beta * (pi_l - ref_l)
    loss = -F.logsigmoid(r_w - r_l).mean()
    return loss, r_w, r_l

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
grad_w_formula = -0.1 * torch.sigmoid(torch.tensor(-0.1 * 4.0)).item()
descent_raises_chosen = True

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def sequence_logprob(logits, targets, mask):
    logp = F.log_softmax(logits, dim=-1)
    tok = torch.gather(logp, -1, targets.unsqueeze(-1)).squeeze(-1)
    return (tok * mask).sum(-1)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def train_toy_dpo(beta, steps=400, lr=0.05, labels=None):
    labels = p_ij if labels is None else labels
    theta = torch.nn.Parameter(ref_logits.clone())
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(steps):
        lp = F.log_softmax(theta, -1)
        m = (lp[i] - ref_lp[i]) - (lp[j] - ref_lp[j])
        loss = -(labels * F.logsigmoid(beta * m) + (1 - labels) * F.logsigmoid(-beta * m)).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return F.softmax(theta.detach(), -1)